# Top-K 采样

> 只在概率最高的 K 个 token 中采样，截断长尾分布。

## 背景
纯随机采样会从长尾分布中选到低概率 token，导致输出质量差。
Top-K 采样只保留概率最高的 K 个 token，重新归一化后采样，
截断长尾同时保留多样性。K=1 退化为贪心，K=V 退化为纯采样。

## 公式
$$\text{TopK}(p, k) = \text{sample}\left(\frac{p \cdot \mathbb{1}[p \geq p_{(k)}]}{\sum_{p_i \geq p_{(k)}} p_i}\right)$$
其中 p_{(k)} 是第 k 大的概率值。

## 复杂度
- 时间：O(V + K log K)，排序 + 采样
- 空间：O(K)
- 典型 K：20-50

## 考察点
- K 的选择：K 太小趋近贪心，太大趋近纯采样
- 与 Top-P 的关系：Top-P 自适应截断，通常更鲁棒
- 实现：torch.topk + 重新归一化 + multinomial


In [ ]:
import torch
import torch.nn.functional as F

def topk_sampling(logits, k=5, temperature=1.0):
    """Top-K 采样: 只保留概率最高的 k 个 token 用于采样。"""
    logits = logits / temperature
    topk_vals, topk_indices = torch.topk(logits, k, dim=-1)
    probs = F.softmax(topk_vals, dim=-1)
    sampled_idx = torch.multinomial(probs, 1)
    return topk_indices[0, sampled_idx[0]]

# 验证: k 个候选概率和为 1
logits = torch.randn(1, 100) * 2
k = 5
topk_vals, _ = torch.topk(logits, k, dim=-1)
probs = F.softmax(topk_vals, dim=-1)
assert abs(probs.sum().item() - 1.0) < 1e-6, "top-k probs must sum to 1"
assert probs.shape == (1, k), f"wrong shape: {probs.shape}"

# 验证采样
sampled = topk_sampling(logits, k=5, temperature=1.0)
assert 0 <= sampled.item() < 100, "token out of vocab"
print(f"✅ TopK: k={k}, 候选概率和={probs.sum().item():.4f}, sampled_token={sampled.item()}")


## 小结
- TopK 的候选集合大小固定 K，对"长尾分布"与"尖锐分布"用同一 K 不够自适应——这正是 TopP 要解决的。
- 实现关键：在**子集**上 softmax 再 gather 回原词表索引，不要对全词表 softmax 再 mask（数值上等价但低效）。

## ✅ 测试验证

In [ ]:
# 验证 Top-K 采样
import torch
import torch.nn.functional as F

logits = torch.randn(100)  # vocab=100
K = 10

# Top-K: 只保留概率最大的 K 个，其余置 0
probs = F.softmax(logits, dim=-1)
topk_values, topk_indices = probs.topk(K)

# 验证: 保留的 K 个是概率最大的
sorted_probs, _ = probs.sort(descending=True)
assert torch.allclose(topk_values, sorted_probs[:K]), "topk should be largest K"

# 验证: 重新归一化后和为 1
topk_probs = torch.zeros_like(probs)
topk_probs[topk_indices] = topk_values
topk_probs = topk_probs / topk_probs.sum()
assert abs(topk_probs.sum().item() - 1.0) < 1e-6, "topk probs should sum to 1"

# 验证: 保留 K 个非零
assert (topk_probs > 0).sum().item() == K, f"should have {K} non-zero"

print(f"✅ TopK 测试通过: 保留最大 K={K} 个，归一化后和为 1")
